# EgyptGPT Inference

Generate hieroglyphic inscriptions from the trained sign-level model, translate them to English, and save to Google Drive.

**Requirements:** Colab GPU runtime.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf EgyptGPT
!git clone --recurse-submodules https://github.com/JLansey/EgyptGPT.git
%cd /content/EgyptGPT
!bash setup.sh

## 2. Mount Google Drive and load checkpoint + data

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

%cd /content/EgyptGPT

DRIVE_ROOT = '/content/drive/MyDrive/EgyptGPT_autoresearch'

# --- Load cached data files from Drive (or prepare from scratch) ---
DATA_DIR = 'data/egypt_char'
DRIVE_DATA = f'{DRIVE_ROOT}/data'
DATA_FILES = ['train.bin', 'val.bin', 'meta.pkl']

cached = all(os.path.exists(os.path.join(DRIVE_DATA, f)) for f in DATA_FILES)
if cached:
    print('Loading cached data from Google Drive...')
    for f in DATA_FILES:
        shutil.copy(os.path.join(DRIVE_DATA, f), os.path.join(DATA_DIR, f))
else:
    print('No cache. Running prepare.py...')
    !cd /content/EgyptGPT && python -c "from data.egypt_char import prepare"
    os.makedirs(DRIVE_DATA, exist_ok=True)
    for f in DATA_FILES:
        shutil.copy(os.path.join(DATA_DIR, f), os.path.join(DRIVE_DATA, f))
    print(f'Cached to {DRIVE_DATA}')

# --- Copy checkpoint from Drive ---
CKPT_SRC = f'{DRIVE_ROOT}/out-glyphs-char/ckpt.pt'
CKPT_DIR = 'out-glyphs-char'
os.makedirs(CKPT_DIR, exist_ok=True)

assert os.path.exists(CKPT_SRC), f'Checkpoint not found on Drive: {CKPT_SRC}'
shutil.copy(CKPT_SRC, os.path.join(CKPT_DIR, 'ckpt.pt'))
print(f'Checkpoint: {os.path.getsize(os.path.join(CKPT_DIR, "ckpt.pt")):,} bytes')

# --- Verify sign-level meta exists ---
SIGN_META = os.path.join(DATA_DIR, 'meta_signs.pkl')
assert os.path.exists(SIGN_META), f'MISSING: {SIGN_META}'
print(f'Sign vocab: {SIGN_META} ({os.path.getsize(SIGN_META):,} bytes)')
print('Ready.')

## 3. Generate inscriptions

Runs `sample_signs.py` to generate Gardiner-notation inscriptions at a few temperatures. Adjust `NUM_SAMPLES` as needed.

In [ ]:
NUM_SAMPLES = 2000
TEMPERATURES = [0.7, 0.8, 1.0]

os.makedirs('out', exist_ok=True)

for temp in TEMPERATURES:
    out_file = f'out/generated_signs_{temp}.txt'
    print(f'\n--- Temperature {temp} ---')
    !python sample_signs.py \
        --num_samples {NUM_SAMPLES} \
        --temperature {temp} \
        --max_new_tokens 200 \
        --out_file {out_file}
    print(f'Saved to {out_file}')

## 4. Translate to English and convert to Unicode hieroglyphs

Loads the hiero-transformer model (M2M100) and translates each generated inscription. Saves a CSV with columns: `gardiner`, `glyph`, `english`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('mattiadc/hiero-transformer')
translation_model = AutoModelForSeq2SeqLM.from_pretrained('mattiadc/hiero-transformer').to(device)
print(f'Translation model loaded on {device}')

In [ ]:
from glyph_utils import gardiner_to_hieroglyphics
from tqdm import tqdm
import pandas as pd

DRIVE_OUTPUT = f'{DRIVE_ROOT}/inference_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

lang_to_m2m = {'ea': 'ar', 'en': 'en'}

def translate_chunk(text, max_length=128):
    """Translate a single Gardiner chunk to English."""
    with torch.no_grad():
        tokenizer.src_lang = lang_to_m2m['ea']
        tokenizer.tgt_lang = lang_to_m2m['en']
        inputs = tokenizer(text, return_tensors='pt').to(translation_model.device)
        tokens = translation_model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.get_lang_id(lang_to_m2m['en']),
            num_beams=10,
            max_length=max_length,
        )
        return tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

def translate_long_text(text, max_chars=128):
    """Split long text into chunks and translate each."""
    words = text.split()
    if not words:
        return ''
    total_chars = sum(len(w) for w in words) + len(words) - 1
    num_chunks = max(1, -(-total_chars // max_chars))
    chars_per_chunk = total_chars / num_chunks

    chunks, current, length, target = [], [], 0, chars_per_chunk
    for word in words:
        if length + len(word) > target and current:
            chunks.append(' '.join(current))
            current, length = [], 0
            target += chars_per_chunk
        current.append(word)
        length += len(word) + 1
    if current:
        chunks.append(' '.join(current))

    return ' '.join(translate_chunk(c) for c in chunks)

for temp in TEMPERATURES:
    in_file = f'out/generated_signs_{temp}.txt'
    out_csv = f'out/translated_{temp}.csv'

    with open(in_file) as f:
        samples = [line.strip() for line in f if line.strip()]
    print(f'\nTemperature {temp}: {len(samples)} inscriptions')

    rows = []
    for i, gardiner in tqdm(enumerate(samples), total=len(samples)):
        glyph = gardiner_to_hieroglyphics(gardiner)
        english = translate_long_text(gardiner)
        rows.append({'gardiner': gardiner, 'glyph': glyph, 'english': english})

        # Periodic save
        if (i + 1) % 200 == 0:
            df = pd.DataFrame(rows)
            df.to_csv(out_csv, index=False)
            df.to_csv(f'{DRIVE_OUTPUT}/translated_{temp}.csv', index=False)
            print(f'  [{i+1}/{len(samples)}] saved checkpoint')

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    df.to_csv(f'{DRIVE_OUTPUT}/translated_{temp}.csv', index=False)
    print(f'Saved {len(df)} rows to {DRIVE_OUTPUT}/translated_{temp}.csv')

## 5. Preview results

In [ ]:
df = pd.read_csv(f'{DRIVE_OUTPUT}/translated_0.8.csv')
print(f'{len(df)} inscriptions')
df.head(20)